In [74]:
import pandas as pd

In [75]:
pois = pd.read_csv('Nodes/N-POIs.csv',dtype={'COD_POSTAL': 'str'})
pois['COD_POSTAL'].nunique()

38

In [76]:
demand = pd.read_csv('Data/mobilitat 2011 codis postals/dest-origen.csv',sep = ';',encoding='latin-1')
demand

,MUNIDEST,CPDESTI,ORIGEN,RAO,PERSONES
0,Badalona,08911,08911,Treball,2812
1,Badalona,08911,08912,Treball,1131
2,Badalona,08911,08913,Treball,673
3,Badalona,08911,08914,Treball,443
4,Badalona,08911,08915,Treball,824
...,...,...,...,...,...
4579,Terrassa,08228,08228,Estudi,683
4580,Terrassa,08228,Terrassa,Estudi,1222
4581,Terrassa,08228,Altres municipis,Estudi,156
4582,Terrassa,08228,Total,Estudi,1378


In [77]:
demand = pd.read_csv('Data/mobilitat 2011 codis postals/dest-origen.csv',sep = ';',encoding='latin-1')
demand = demand[demand['MUNIDEST'] == 'Barcelona']
demand = demand[demand['CPDESTI'] != '00Barcelona']
#demand = demand[demand['ORIGEN'] == 'Mataró']
demand['PERSONES'] = demand['PERSONES'].str.replace('..','0')
demand['PERSONES'] = demand['PERSONES'].astype(int)
demand_grouped = demand.groupby(['CPDESTI']).agg({'PERSONES':'sum'}).reset_index()
demand_grouped['PCT'] = demand_grouped['PERSONES'] / demand_grouped['PERSONES'].sum() * 100
demand_grouped

,CPDESTI,PERSONES,PCT
0,08001,134101,4.399208
1,08002,78734,2.582883
2,08003,96898,3.178756
3,08004,54355,1.783126
4,08005,103676,3.401110
5,08006,105535,3.462095
6,08007,87951,2.885248
7,08008,57394,1.882821
8,08009,61560,2.019487
9,08010,70064,2.298462


In [78]:
pois["N"] = pois.groupby("COD_POSTAL")["COD_POSTAL"].transform("size")

# Merge demand by postal code
pois = pois.merge(
    demand_grouped,
    left_on="COD_POSTAL",
    right_on="CPDESTI",
    how="left"
)

pois["PCT"] = pois["PCT"].div(pois["N"])
pois = pois.drop(columns=["N", "CPDESTI"])

In [79]:
demand_full = pd.DataFrame()

# Mataro e11.1/2

In [80]:
FREQ = 4.4
H_SERVEI = 5
BUS_CAPACITY = 80
total_demand = FREQ * H_SERVEI * BUS_CAPACITY
total_demand

1760.0

In [81]:
demand_e11 = pois.copy()
demand_e11.drop(columns=['name','geometry','barri','COD_POSTAL','PERSONES'], inplace=True)
demand_e11['demand'] = demand_e11['PCT'] *  total_demand / 100
demand_e11.drop(columns=['PCT'], inplace=True)
demand_e11 = (
    demand_e11
    .set_index("id")["demand"]
    .to_frame()
    .T
    .reset_index(drop=True)
)
demand_e11.index = ['E11']
demand_e11

id,1,2,3,4,5,6,7,8,9,10,...,89,90,91,92,93,94,95,96,97,98
E11,25.633548,11.971445,40.452935,16.926791,11.971907,22.347152,14.727873,2.85895,9.324352,25.633548,...,16.081327,56.267709,25.808684,31.383011,11.14182,30.892245,33.293532,23.820026,30.708064,4.60059


In [82]:
demand_full = pd.concat([demand_full, demand_e11])
demand_full

id,1,2,3,4,5,6,7,8,9,10,...,89,90,91,92,93,94,95,96,97,98
E11,25.633548,11.971445,40.452935,16.926791,11.971907,22.347152,14.727873,2.85895,9.324352,25.633548,...,16.081327,56.267709,25.808684,31.383011,11.14182,30.892245,33.293532,23.820026,30.708064,4.60059
